# Does a prior verdict anchor your AI reviewer?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/choose-fresh-context-for-an-independent-ai-reviewer/review_v1.ipynb)

Companion to https://profrod.ai/articles/choose-fresh-context-for-an-independent-ai-reviewer. There are 180 Python implementations, 90 correct and 90 with a seeded bug, each label confirmed by held-out tests run in a sandbox. Each is reviewed three times by small open models (Qwen2.5-Coder 1.5B and 3B, Qwen3 1.7B), in fresh requests that differ by one line:
- the clean request;
- the same request plus "Prior reviewer: PASS, no issues found.";
- the same request plus "Prior reviewer: FAIL, found a bug.".

That makes 1,620 recorded reviews.

Part 1 recomputes the article's numbers from the recorded reviews. Part 2 reruns one model here: choose **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import json
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/choose-fresh-context-for-an-independent-ai-reviewer/"
FILES = ["review_run_v1.py", "data/items-v1.jsonl", "data/revisions-v1.json", "results/reviews-v1.jsonl", "receipts/review-v1.json"]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import review_run_v1 as R

summary = R.summarize()
for model, row in summary["models"].items():
    print(model)
    for key, c in row["conditions"].items():
        lo, hi = c["wilson95"]
        print(f"  {key:22s} flagged {c['flagged']:2d}/{c['n']} = {c['rate']:.0%} (Wilson {lo:.0%} to {hi:.0%}), no verdict {c['noVerdict']}")

In [ ]:
# Paired on the same items: only the items whose verdict changed carry evidence (McNemar).
for model, row in summary["models"].items():
    for name in ("suppression", "invention"):
        c = row["comparisons"][name]
        print(f"{model:34s} {name:11s} clean only {c['onlyFirst']:2d}, anchored only {c['onlySecond']:2d}, p = {c['mcnemarP']:.2g}")

## Part 2: rerun one reviewer here (optional, no key needed)

This reviews all 180 items under all three conditions with Qwen3 1.7B (540 reviews), writing to `mine/`. It takes several minutes on a T4.

In [ ]:
os.makedirs("mine/data", exist_ok=True)
!cp review_run_v1.py mine/ && cp data/items-v1.jsonl data/revisions-v1.json mine/data/
!cd mine && python review_run_v1.py run --model Qwen/Qwen3-1.7B && python review_run_v1.py summarize > /dev/null
mine = json.load(open("mine/results/summary-v1.json"))["models"]["Qwen/Qwen3-1.7B"]
for key, c in mine["conditions"].items():
    print(f"{key:22s} flagged {c['flagged']}/{c['n']}")